# APIs & Real Data — From Fetch to Analysis
### Pulling, cleaning and exploring data in the real world

---

## What is an API?

So far we always loaded data from a file on our computer:

```python
covid_data = pd.read_csv('covid.csv')   # file on your hard drive
```

The problem: that file gets **stale**. If the data updates tomorrow, your file does not.

An **API (Application Programming Interface)** lets your code talk directly to a
server and pull **fresh data** every time you run it — no downloading, no manual updates.

```
Without API:   your code  →  local CSV file   (frozen in time)
With API:      your code  →  internet  →  live server  →  fresh data
```

---

### How does an API call work?

When you type a URL in a browser, your browser sends a **request** to a server
and gets back a **response** — normally an HTML page.

An API works exactly the same way — but instead of HTML, the server sends back
**data** that Python can read directly.

```
Step 1  →  Python sends a GET request to a URL
Step 2  →  Server receives the request
Step 3  →  Server sends back data
Step 4  →  Python reads it into a DataFrame
```

The Python library that handles this is called **requests**.

---

### Two formats you will encounter

| Format | What it looks like | How to read it |
|--------|-------------------|----------------|
| **CSV** | Comma-separated table — same as a spreadsheet | `pd.read_csv(url)` |
| **JSON** | Nested key-value pairs — like a Python dictionary | `response.json()` + `pd.json_normalize()` |


---

### Status codes — always check this first

| Code | Meaning |
|------|---------|
| 200 | ✅ Success |
| 400 | ❌ Bad request — something wrong with your URL |
| 401 | 🔑 Unauthorised — you need an API key |
| 404 | ❌ Not found — wrong URL |
| 500 | ❌ Server error — their problem, not yours |

kxmkwmxkwmxkwxw


In [ ]:
jxnenmkewmxkemceslknxje xj jx wjxwkw,xlw,x,wlx,wl,xlw,lxwjxnjxkwxkw xjw xkjwkjxwmx,lwö,ölw,q


In [ ]:
!pip install requests

In [ ]:
import requests          # sends HTTP requests
import pandas as pd      # DataFrames
import io                # converts raw response text into something pandas can read
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print('Ready!')

---
## Part 1 — CSV from a URL

When an API returns a CSV file, there are two ways to load it.

**Method 1 — pandas reads the URL directly (simplest)**
```python
data = pd.read_csv(url, storage_options=headers)
```

**Method 2 — requests then pandas (more control)**
```python
response = requests.get(url, headers=headers)
print(response.status_code)                      # check it worked first
data = pd.read_csv(io.StringIO(response.text))   # convert to DataFrame
```

> **Why `io.StringIO()`?**  
> `response.text` is a plain string — a giant block of text.  
> `pd.read_csv()` expects a file, not a string.  
> `io.StringIO()` wraps the string so pandas thinks it is reading a file.

> ⚠️ Some APIs block automated requests without a browser-like header.  
> The fix is always the same — pass `headers = {'User-Agent': 'Mozilla/5.0'}`.  
> GitHub Raw URLs (like the COVID dataset) do not need this. OWID chart URLs do.

### Worked example — Life expectancy (OWID)

Every chart on ourworldindata.org becomes a data URL by adding `.csv` to the URL.

In [ ]:
# Method 1 — simplest
url_life = 'https://ourworldindata.org/grapher/life-expectancy.csv'
headers  = {'User-Agent': 'Mozilla/5.0'}

life_data = pd.read_csv(url_life, storage_options=headers)

rows     = life_data.shape[0]
cols     = life_data.shape[1]
columns  = life_data.columns.tolist()

print(f'Rows    : {rows:,}')
print(f'Columns : {cols}')
print(f'Names   : {columns}')

In [ ]:
# Method 2 — explicit request
# This makes the HTTP request visible so we can inspect it before loading

response = requests.get(url_life, headers=headers)

status = response.status_code
size   = len(response.text)

print(f'Status code   : {status}')   # 200 = OK
print(f'Response size : {size:,} characters')
print()
print('First 300 characters of raw response:')
print(response.text[:300])

In [ ]:
# Now convert the raw response to a DataFrame
life_data = pd.read_csv(io.StringIO(response.text))

rows      = life_data.shape[0]
countries = life_data['Entity'].nunique()

print(f'Rows      : {rows:,}')
print(f'Countries : {countries}')

### Exercise 1 — CSV from a URL

> **The OWID pattern:** any chart on ourworldindata.org → add `.csv` to the URL → data

In [ ]:
# Q1: Load the CO2 emissions dataset using Method 1
# hint: same pattern as life expectancy — just swap the URL

url_co2 = 'https://ourworldindata.org/grapher/annual-co2-emissions-per-country.csv'
headers = {'User-Agent': 'Mozilla/5.0'}

co2_data = pd.read_csv(___, storage_options=___)   # fill in: url_co2, headers

print(co2_data.shape)

# ANSWER: pd.read_csv(url_co2, storage_options=headers)

In [ ]:
# Q2: Load the same dataset using Method 2
# hint: requests.get(url, headers=headers) then check .status_code before reading

response_co2 = requests.get(___, headers=___)   # fill in: url_co2, headers

status = response_co2.___                        # hint: .status_code
print(f'Status: {status}')

co2_data = pd.read_csv(io.StringIO(response_co2.___))   # hint: .text
print(co2_data.columns.tolist())

# ANSWER: requests.get(url_co2, headers=headers) / .status_code / .text

---
## Part 2 — JSON API

Many APIs return **JSON** instead of CSV. JSON is not a table — it is a
nested structure of keys and values, like a Python dictionary.

```json
{
  "name": "Brazil",
  "population": 215313498,
  "address": {
      "city": "Brasilia",
      "region": "South America"
  }
}
```

You cannot `pd.read_csv()` this — you need two steps:

```python
# Step 1: parse JSON into a Python list or dict
data = response.json()

# Step 2: flatten the nested structure into a DataFrame
json_data = pd.json_normalize(data)
```

`pd.json_normalize()` flattens nested keys into column names using dot notation:  
`address.city`, `address.region`.

### Worked example — JSONPlaceholder

A free API that returns realistic fake data. No key, no sign-up, never goes down.

In [ ]:
# Fetch user data — returns a list of 10 users in JSON format
url_users = 'https://jsonplaceholder.typicode.com/users'

response = requests.get(url_users)

status = response.status_code
print(f'Status: {status}')
print()
print('Raw response (first 400 characters) — notice it is NOT a table:')
print(response.text[:400])

In [ ]:
# Step 1 — parse JSON into a Python list
users_json = response.json()

kind   = type(users_json)
length = len(users_json)

print(f'Type   : {kind}')
print(f'Length : {length}')
print()

# Look at ONE user to see the nested structure before flattening
import json
print(json.dumps(users_json[0], indent=2))

In [ ]:
# Step 2 — flatten into a DataFrame
# nested keys become columns: address.city, company.name
users_data = pd.json_normalize(users_json)

print('All columns after flattening:')
print(users_data.columns.tolist())

In [ ]:
# Select only the columns we want
users_clean = users_data[['name', 'email', 'address.city', 'company.name']].copy()
users_clean.columns = ['name', 'email', 'city', 'company']

print(users_clean.to_string(index=False))

### Exercise 2 — JSON API

In [ ]:
# JSONPlaceholder also has a posts endpoint with 100 blog posts
# https://jsonplaceholder.typicode.com/posts

# Q1: Fetch the posts endpoint and check status
url_posts      = 'https://jsonplaceholder.typicode.com/posts'
response_posts = requests.get(___)              # hint: url_posts

status = response_posts.___                     # hint: .status_code
print(f'Status: {status}')

# Q2: Parse JSON and flatten into a DataFrame
posts_json = response_posts.___()               # hint: .json()
posts_data = pd.json_normalize(___)             # hint: posts_json

print(f'Shape   : {posts_data.shape}')
print(f'Columns : {posts_data.columns.tolist()}')

# Q3: Show the title of the first 5 posts
print(posts_data[['___']].head())

# ANSWERS:
# Q1: requests.get(url_posts) / .status_code
# Q2: .json() / posts_json
# Q3: posts_data[['title']].head()

---
## Part 3 — COVID-19: understand and clean a real dataset

Now we apply what we learned. The COVID dataset from Our World in Data is
a perfect example of real-world messiness:

- The `location` column mixes countries, continents and world aggregates
- The `total_cases` column looks useful but is not reliable
- Missing values need decisions before we can calculate anything

We will discover these problems through the data — not assume them upfront.

### Step 1 — Pull the data

In [ ]:
# Pull the COVID dataset from Our World in Data
url_covid = 'https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv'

response  = requests.get(url_covid)

status = response.status_code
print(f'Status: {status}')

covid_raw = pd.read_csv(io.StringIO(response.text))

rows = covid_raw.shape[0]
cols = covid_raw.shape[1]
print(f'Rows    : {rows:,}')
print(f'Columns : {cols}')

### Step 2 — Understand the data

Before cleaning anything, we look at what we have.

In [ ]:
# Overview — shape, date range, column types
rows     = covid_raw.____[0]
cols     = covid_raw.____[1]
date_min = covid_raw['date'].____()
date_max = covid_raw['date'].___()

print(f'Rows     : {rows:,}')
print(f'Columns  : {cols}')
print(f'Date min : {date_min}')
print(f'Date max : {date_max}')

In [ ]:
# Column types
print(covid_raw.dtypes)

### Step 3 — Check missing values

Always check missing values before touching anything — it gives the full picture.

In [ ]:
# How many missing values in total?
# hint: .isnull().sum().sum() — first sum per column, then sum all columns

total_missing = covid_raw.___
print(total_missing)






In [ ]:
# Which columns have missing values?
# hint: .isnull().sum() gives missing count per column
# then filter to only columns where missing > 0

missing = covid_raw.___
missing = _____[________ > ___]
print(missing)

# ANSWER: covid_raw.isnull().sum() / missing > 0

In [ ]:
# What percentage of each column is missing?
# hint: divide missing by total rows, multiply by 100

missing_pct = (missing / ___ * 100).round(1)
print(missing_pct)



### Step 4 — Investigate the location problem

`continent` has missing values — let's find out why.

In [ ]:
# How many rows have a null continent?
null_rows = covid_raw['continent'].isnull().sum()


print(f'Rows with null continent: {null_rows:,}')
print()

# What locations appear in those rows?
non_countries = covid_raw[covid_raw['continent'].isnull()]['location'].unique()

print('Locations with null continent:')
for loc in sorted(non_countries):
    print(f'  {loc}')

These are **aggregates** — World, Europe, High income, etc.  
Every real country belongs to a continent. These do not.  
We will remove them by keeping only rows where `continent` is NOT null.

---
## Part 4 — Clean the data

Now we fix everything step by step.

```python
# Key formulas:
data[['col1', 'col2']].copy()          # keep only selected columns
data[data['col'].notna()]              # keep rows where column is NOT null
pd.to_datetime(data['date'])           # convert string to datetime
data['date'].dt.year                   # extract year from datetime
data['col'].fillna(0)                  # replace NaN with 0
```

In [ ]:
# Step 1 — keep only the columns we need
# We have 67 columns but only need 6 for our analysis

columns_needed = ['location', 'continent', 'date', 'new_cases', 'new_deaths', 'population']
covid_clean    = covid_raw[columns_needed].copy()

cols_before = covid_raw.shape[1]
cols_after  = covid_clean.shape[1]

print(f'Columns before : {cols_before}')
print(f'Columns after  : {cols_after}')

In [ ]:
# Step 2 — remove aggregate rows
# Keep only rows where continent is NOT null — these are real countries
# hint: .notna() returns True for rows that are NOT null

covid_clean = _____[__________['_______'].___()].copy()

rows      = len(covid_clean)
countries = covid_clean['location'].nunique()

print(f'Rows remaining : {rows:,}')
print(f'Countries      : {countries}')

# ANSWER: .notna()

In [ ]:
# Step 3 — convert date from string to datetime
# hint: pd.to_datetime() converts a string column to datetime
# Without this we cannot filter by year or sort properly

covid_clean['date'] = pd.___(___________['_______'])   

date_type = covid_clean['date'].dtype
print(date_type)



In [ ]:
# Step 4 — filter to 2020–2023
# hint: .dt.year extracts the year from a datetime column

covid_clean = covid_clean[
    (covid_clean['date'].dt.year >= 2020) &   # fill in: .dt
    (covid_clean['date'].dt.year <= 2023)
].copy()

rows     = len(covid_clean)
date_min = covid_clean['date'].min().date()
date_max = covid_clean['date'].max().date()

print(f'Rows     : {rows:,}')
print(f'Date min : {date_min}')
print(f'Date max : {date_max}')



In [ ]:
# Step 5 — check missing values before filling
# hint: .isnull().sum() counts missing values in a column

missing_cases  = covid_clean['new_cases'].___.___   # fill in
missing_deaths = covid_clean['new_deaths'].___.___  # fill in

print('Missing values BEFORE:')
print(f'  new_cases  : {missing_cases:,}')
print(f'  new_deaths : {missing_deaths:,}')

# ANSWER: .isnull().sum()

In [ ]:
# Now fill missing values with 0
# hint: .fillna(0) replaces all NaN with 0

covid_clean['new_cases']  = covid_clean['new_cases'].___(0)    # fill in
covid_clean['new_deaths'] = covid_clean['new_deaths'].___(0)   # fill in

# ANSWER: .fillna(0)

In [ ]:
# Check after — both should show 0
missing_cases  = covid_clean['new_cases'].isnull().sum()
missing_deaths = covid_clean['new_deaths'].isnull().sum()

print('Missing values AFTER:')
print(f'  new_cases  : {missing_cases}')
print(f'  new_deaths : {missing_deaths}')

---
## Part 5 — Build reliable metrics

Now we create the columns we will actually use for analysis.

```python
# Key formulas:
data.sort_values(['location', 'date'])                  # sort before cumsum
data.groupby('location')['new_cases'].cumsum()          # cumulative per country
(data['new_cases'] / data['population']) * 1_000_000    # per million
```

In [ ]:
# Step 6a — sort by location and date
# This is required before cumsum
# The running total must go in chronological order and restart for each country

covid_clean = covid_clean.sort_values(['location', 'date'])
print('Sorted by location and date.')

In [ ]:
# Step 6b — build our own cumulative cases and deaths
# groupby('location') makes cumsum restart for each country
# without this, the running total would continue across countries

covid_clean['cumulative_cases']  = covid_clean.groupby('location')['new_cases'].cumsum()
covid_clean['cumulative_deaths'] = __________.__________('__________')['___________']._____()

# Verify with Brazil — filter to rows where cases actually started
brazil       = covid_clean[covid_clean['location'] == 'Brazil'].copy()
brazil_cases = brazil[brazil['new_cases'] > 0]

print(brazil_cases[['date', 'new_cases', 'cumulative_cases']].head(10).to_string(index=False))

In [ ]:
# Step 6c — cases and deaths per million
# Raw numbers are not fair for comparison
# A country with 1 billion people will always have more cases than one with 5 million
# Per million puts everyone on the same scale

covid_clean['new_cases_per_million']  = (covid_clean['new_cases']  / covid_clean['population']) * 1_000_000
covid_clean['new_deaths_per_million'] = (___________['___________'] / __________['_______']) * _______

print('All columns now available:')
print(covid_clean.columns.tolist())

### Exercise 3 — Clean the data

In [ ]:
# Q1: How many unique countries are in covid_clean?
# hint: .nunique() counts unique values

print(_______['_______'].___())



In [ ]:
# Q2: Filter to one country of your choice and show 5 rows
my_country   = '___'
country_data = covid_clean[covid_clean['location'] == ___]
print(country_data.head(5).to_string(index=False))

# ANSWER: covid_clean[covid_clean['location'] == my_country]

In [ ]:
# Q3: What is Brazil's total cumulative cases by end of 2023?

# Step 1 — filter to Brazil
brazil = covid_clean[covid_clean['location'] == 'Brazil']

rows = len(brazil)
print(f'Brazil rows: {rows}')

In [ ]:
# Step 2 — find the last date available
last_date = brazil['date'].___   # hint: .max()
print(last_date)

# ANSWER: .max()

In [ ]:
# Step 3 — filter to that date and read the cumulative total
last_row = brazil[brazil['date'] == last_date]
print(last_row[['date', 'new_cases', 'cumulative_cases']])

---
## Part 6 — Simple KPIs

```python
# Key formulas:
data['new_cases'].max()                           # peak value
data.loc[data['new_cases'].idxmax(), 'date']      # date of peak
data.groupby('continent')['new_cases'].sum()      # total by group
```

In [ ]:
# KPI 1 — Daily new cases for Brazil (raw)
country_name = 'Brazil'
country_data = covid_clean[covid_clean['location'] == country_name].copy()

plt.figure(figsize=(11, 4))
plt.plot(country_data['date'], country_data['new_cases'], color='#378ADD', linewidth=0.8)
plt.xlabel('Date')
plt.ylabel('New cases')
plt.title(f'{country_name} — Daily new cases 2020–2023')
plt.tight_layout()
plt.show()

In [ ]:
# KPI 2 — Total cases by continent
by_continent = covid_clean.groupby('continent')['new_cases'].sum().sort_values(ascending=False)
print(by_continent)

In [ ]:
# KPI 3 — Top 10 countries by total cases
by_country = covid_clean.groupby('location')['new_cases'].sum().sort_values(ascending=False)
print(by_country.head(10))

In [ ]:
# KPI 4 — Deaths per million over time — compare countries
brazil  = covid_clean[covid_clean['location'] == 'Brazil'].copy()
usa     = covid_clean[covid_clean['location'] == 'United States'].copy()
germany = covid_clean[covid_clean['location'] == 'Germany'].copy()

plt.figure(figsize=(12, 4))
plt.plot(brazil['date'],  brazil['new_deaths_per_million'],  linewidth=1.5, label='Brazil')
plt.plot(usa['date'],     usa['new_deaths_per_million'],     linewidth=1.5, label='United States')
plt.plot(germany['date'], germany['new_deaths_per_million'], linewidth=1.5, label='Germany')

plt.xlabel('Date')
plt.ylabel('Deaths per million')
plt.title('COVID-19 — Deaths per million 2020–2023')
plt.legend()
plt.tight_layout()
plt.show()

### Exercise 4 — Your turn

In [ ]:
# Q1 — Step 1: filter to your country and plot daily new deaths
my_country = '___'
my_data    = covid_clean[covid_clean['location'] == ___].copy()

plt.figure(figsize=(11, 4))
plt.plot(_____['date'], ____________['new_deaths'], color='#e74c3c', linewidth=0.8)
plt.xlabel('Date')
plt.ylabel('New deaths')
plt.title(f'{my_country} — Daily new deaths 2020–2023')
plt.tight_layout()
plt.show()

In [ ]:
# Q2: Top 5 countries by total deaths
# hint: .sum() then .sort_values(ascending=False)

by_deaths = covid_clean.groupby('location')['new_deaths'].___().sort_values(ascending=___)
print(by_deaths.head(5))







---

## Suggested next calculations

| Calculation | How |
|-------------|-----|
| Case fatality rate | `new_deaths.sum() / new_cases.sum()` per country |
| Monthly totals | `groupby` by location and month |
| Year-over-year comparison | Filter by `date.dt.year` then compare |
| Peak week per country | `rolling(7).mean().idxmax()` per country |
| Deaths per million by continent | `groupby('continent')` on `new_deaths_per_million` |

---
## Where to find more data

### The OWID pattern — every chart is a dataset

```
Chart page:  https://ourworldindata.org/grapher/SLUG
Data (CSV):  https://ourworldindata.org/grapher/SLUG.csv
```

Go to https://ourworldindata.org, find any topic, copy the URL slug — that is all you need.

| Dataset | URL |
|---------|-----|
| Life expectancy | `https://ourworldindata.org/grapher/life-expectancy.csv` |
| CO₂ emissions | `https://ourworldindata.org/grapher/annual-co2-emissions-per-country.csv` |
| Population | `https://ourworldindata.org/grapher/population.csv` |
| Child mortality | `https://ourworldindata.org/grapher/child-mortality.csv` |
| GDP per capita | `https://ourworldindata.org/grapher/gdp-per-capita-worldbank.csv` |

---

### Other free APIs — JSON format

| API | URL | What you get |
|-----|-----|--------------|
| JSONPlaceholder | `https://jsonplaceholder.typicode.com/users` | Fake user data for practice |
| Open Meteo | `https://api.open-meteo.com/v1/forecast?latitude=52.52&longitude=13.41&daily=temperature_2m_max` | Weather forecast |
| NASA APOD | `https://api.nasa.gov/planetary/apod?api_key=DEMO_KEY` | Astronomy picture of the day |
| CoinGecko | `https://api.coingecko.com/api/v3/coins/bitcoin/market_chart?vs_currency=usd&days=30` | Bitcoin price last 30 days |

---



## Quick Reference

```python
# ── CSV from URL ──────────────────────────────────────────────────────────────
data = pd.read_csv(url)                               # Method 1
response = requests.get(url)                          # Method 2
print(response.status_code)                           # 200 = OK
data = pd.read_csv(io.StringIO(response.text))

# ── JSON from URL ─────────────────────────────────────────────────────────────
response  = requests.get(url)
raw_json  = response.json()                           # parse JSON
json_data = pd.json_normalize(raw_json)               # flatten to DataFrame

# ── Clean ─────────────────────────────────────────────────────────────────────
data[['col1', 'col2']].copy()                         # keep selected columns
data[data['continent'].notna()]                       # remove null rows
pd.to_datetime(data['date'])                          # string to datetime
data['date'].dt.year                                  # extract year
data['col'].fillna(0)                                 # fill nulls with 0

# ── Build metrics ─────────────────────────────────────────────────────────────
data.groupby('location')['new_cases'].cumsum()        # cumulative per country
(data['new_cases'] / data['population']) * 1_000_000  # per million

# ── Analyse ───────────────────────────────────────────────────────────────────
data['new_cases'].rolling(7).mean()                   # 7-day rolling average
data['new_cases'].max()                               # peak value
data.loc[data['new_cases'].idxmax(), 'date']          # date of peak
data.groupby('continent')['new_cases'].sum()          # total by group
```